# 03 — Model Training & Evaluation
**NIDS — Network Intrusion Detection System**

This notebook:
1. Loads preprocessed data from `data/processed/`
2. Trains an XGBoost multi-class classifier
3. Evaluates on Validation + Test sets
4. Generates SHAP feature importance
5. Tracks everything with MLflow
6. Saves `models/model.pkl` (required by the FastAPI backend)
7. Saves `models/model_info.json` (serves the Model Info page)

## 0. Imports & Paths

In [1]:
import warnings
warnings.filterwarnings('ignore')

import os, json
import numpy as np
import pandas as pd
import joblib
import matplotlib
matplotlib.use('Agg')   # non-interactive backend — avoids Tk/display errors
import matplotlib.pyplot as plt
import seaborn as sns

from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, f1_score,
    classification_report, confusion_matrix
)

import shap
import mlflow
import mlflow.xgboost

# ── Paths (relative to notebooks/ folder) ─────────────────────
BASE_DIR      = os.path.dirname(os.getcwd())   # resolves to NIDS/ when run from notebooks/
PROCESSED_DIR = os.path.join(BASE_DIR, 'data', 'processed')
MODELS_DIR    = os.path.join(BASE_DIR, 'models')
os.makedirs(MODELS_DIR, exist_ok=True)

# ── MLflow tracking URI ────────────────────────────────────────
mlflow.set_tracking_uri(
    'sqlite:///' + os.path.join(BASE_DIR, 'mlruns', 'mlflow.db').replace('\\', '/')
)

print('Processed dir:', PROCESSED_DIR)
print('Models dir   :', MODELS_DIR)
print('Files in processed:', os.listdir(PROCESSED_DIR))

Processed dir: d:\NIDS\data\processed
Models dir   : d:\NIDS\models
Files in processed: ['feature_names.pkl', 'label_encoders.pkl', 'scaler.pkl', 'target_encoder.pkl', 'X_test.npy', 'X_train.npy', 'X_val.npy', 'y_test.npy', 'y_train.npy', 'y_val.npy']


## 1. Load Preprocessed Data

In [2]:
X_train = np.load(os.path.join(PROCESSED_DIR, 'X_train.npy'))
y_train = np.load(os.path.join(PROCESSED_DIR, 'y_train.npy'))
X_val   = np.load(os.path.join(PROCESSED_DIR, 'X_val.npy'))
y_val   = np.load(os.path.join(PROCESSED_DIR, 'y_val.npy'))
X_test  = np.load(os.path.join(PROCESSED_DIR, 'X_test.npy'))
y_test  = np.load(os.path.join(PROCESSED_DIR, 'y_test.npy'))

feature_names  = joblib.load(os.path.join(PROCESSED_DIR, 'feature_names.pkl'))
target_encoder = joblib.load(os.path.join(PROCESSED_DIR, 'target_encoder.pkl'))
CLASS_NAMES    = list(target_encoder.classes_)

# Cast to float32 — XGBoost 3.x prefers it and it avoids dtype warnings
X_train = X_train.astype(np.float32)
X_val   = X_val.astype(np.float32)
X_test  = X_test.astype(np.float32)

print(f'X_train : {X_train.shape}   y_train : {y_train.shape}')
print(f'X_val   : {X_val.shape}   y_val   : {y_val.shape}')
print(f'X_test  : {X_test.shape}   y_test  : {y_test.shape}')
print(f'Features: {len(feature_names)}')
print(f'Classes : {CLASS_NAMES}')

X_train : (269370, 40)   y_train : (269370,)
X_val   : (25195, 40)   y_val   : (25195,)
X_test  : (22544, 40)   y_test  : (22544,)
Features: 40
Classes : ['DoS', 'Normal', 'Probe', 'R2L', 'U2R']


## 2. Train XGBoost Classifier

> **Why XGBoost?**  
> NSL-KDD is tabular and moderately sized. XGBoost is the standard choice:
> fast, handles class imbalance well (we already SMOTE'd), gives feature
> importances natively, and is what `ml.py` already expects.

In [3]:
# ── MLflow experiment ──────────────────────────────────────────
_experiment = mlflow.get_experiment_by_name('NIDS-XGBoost')
if _experiment is None:
    mlflow.create_experiment('NIDS-XGBoost')
mlflow.set_experiment('NIDS-XGBoost')

with mlflow.start_run(run_name='xgb_baseline') as run:

    params = dict(
        n_estimators      = 300,
        max_depth         = 6,
        learning_rate     = 0.1,
        subsample         = 0.8,
        colsample_bytree  = 0.8,
        min_child_weight  = 5,
        gamma             = 0.1,
        objective         = 'multi:softprob',
        num_class         = len(CLASS_NAMES),
        eval_metric       = 'mlogloss',
        random_state      = 42,
        n_jobs            = -1,
        tree_method       = 'hist',
    )
    mlflow.log_params(params)

    model = XGBClassifier(**params)
    model.fit(
        X_train, y_train,
        eval_set  = [(X_val, y_val)],
        verbose   = 50,
    )

    y_val_pred  = model.predict(X_val)
    val_acc     = accuracy_score(y_val, y_val_pred)
    val_f1      = f1_score(y_val, y_val_pred, average='weighted')
    mlflow.log_metric('val_accuracy',    val_acc)
    mlflow.log_metric('val_f1_weighted', val_f1)
    print(f'\n\u2713 Validation  |  Accuracy: {val_acc:.4f}  |  F1 (weighted): {val_f1:.4f}')

    y_test_pred = model.predict(X_test)
    test_acc    = accuracy_score(y_test, y_test_pred)
    test_f1     = f1_score(y_test, y_test_pred, average='weighted')
    mlflow.log_metric('test_accuracy',    test_acc)
    mlflow.log_metric('test_f1_weighted', test_f1)
    print(f'\u2713 Test        |  Accuracy: {test_acc:.4f}  |  F1 (weighted): {test_f1:.4f}')

    RUN_ID = run.info.run_id

print(f'\nMLflow Run ID: {RUN_ID}')

[0]	validation_0-mlogloss:1.38802
[50]	validation_0-mlogloss:0.02302
[100]	validation_0-mlogloss:0.00455
[150]	validation_0-mlogloss:0.00303
[200]	validation_0-mlogloss:0.00270
[250]	validation_0-mlogloss:0.00260
[299]	validation_0-mlogloss:0.00257

✓ Validation  |  Accuracy: 0.9992  |  F1 (weighted): 0.9992
✓ Test        |  Accuracy: 0.8022  |  F1 (weighted): 0.7724

MLflow Run ID: 906238d6d3314f3195e23197ff6c3caa


## 3. Evaluation — Classification Report

In [4]:
print('=' * 55)
print('  TEST SET — Classification Report')
print('=' * 55)
print(classification_report(
    y_test, y_test_pred,
    target_names = CLASS_NAMES
))

  TEST SET — Classification Report
              precision    recall  f1-score   support

         DoS       0.96      0.84      0.90      7458
      Normal       0.71      0.97      0.82      9711
       Probe       0.88      0.81      0.84      2421
         R2L       0.99      0.14      0.25      2754
         U2R       0.71      0.14      0.23       200

    accuracy                           0.80     22544
   macro avg       0.85      0.58      0.61     22544
weighted avg       0.84      0.80      0.77     22544



## 4. Confusion Matrix

In [5]:
cm = confusion_matrix(y_test, y_test_pred)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
    linewidths=0.5, ax=ax
)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual',    fontsize=12)
ax.set_title('Confusion Matrix — Test Set', fontsize=14, fontweight='bold')
plt.tight_layout()

cm_path = os.path.join(MODELS_DIR, 'confusion_matrix.png')
plt.savefig(cm_path, dpi=150)
plt.show()
print(f'Saved \u2192 {cm_path}')

Saved → d:\NIDS\models\confusion_matrix.png


## 5. Feature Importance (XGBoost built-in)

In [6]:
importances = model.feature_importances_
top_n       = 20
top_idx     = np.argsort(importances)[::-1][:top_n]
top_names   = [feature_names[i] for i in top_idx]
top_vals    = importances[top_idx]

fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.barh(top_names[::-1], top_vals[::-1], color='steelblue', edgecolor='white')
ax.set_xlabel('Importance Score', fontsize=12)
ax.set_title(f'Top {top_n} Feature Importances (XGBoost gain)', fontsize=13, fontweight='bold')
ax.bar_label(bars, fmt='%.4f', padding=3, fontsize=8)
plt.tight_layout()

fi_path = os.path.join(MODELS_DIR, 'feature_importance.png')
plt.savefig(fi_path, dpi=150)
plt.show()
print(f'Saved \u2192 {fi_path}')

Saved → d:\NIDS\models\feature_importance.png


## 6. SHAP Explainability

> SHAP (SHapley Additive exPlanations) gives a theoretically sound measure
> of each feature's contribution to every prediction.
> We use a background sample to speed up computation.
>
> **Note:** SHAP >= 0.46 returns shape `(n_samples, n_features, n_classes)`
> for multi-class XGBoost (axes 0 and 1 swapped vs older versions).

In [7]:
# Use a random sample to keep computation fast
rng      = np.random.default_rng(42)
bg_idx   = rng.choice(len(X_train), size=min(500, len(X_train)), replace=False)
X_bg     = X_train[bg_idx]

test_idx = rng.choice(len(X_test), size=min(1000, len(X_test)), replace=False)
X_shap   = X_test[test_idx]

explainer   = shap.TreeExplainer(model, data=X_bg)
shap_values = explainer.shap_values(X_shap)

# Normalise to a 3-D array.
# SHAP >= 0.46 with XGBoost multiclass → (n_samples, n_features, n_classes)
# Older SHAP                           → list of (n_samples, n_features)
shap_arr = np.array(shap_values)  # always makes a plain ndarray

if shap_arr.ndim == 3 and shap_arr.shape[2] == len(CLASS_NAMES):
    # New layout: (n_samples, n_features, n_classes)
    # Transpose to (n_classes, n_samples, n_features) for uniform downstream use
    shap_arr = shap_arr.transpose(2, 0, 1)
elif shap_arr.ndim == 3 and shap_arr.shape[0] == len(CLASS_NAMES):
    # Already (n_classes, n_samples, n_features) — old layout
    pass
else:
    raise ValueError(f'Unexpected shap_values shape: {shap_arr.shape}')

# shap_arr is now reliably (n_classes, n_samples, n_features)
print(f'SHAP values shape (n_classes, n_samples, n_features): {shap_arr.shape}')

Background dataset has 500 samples but max_samples=100. Subsampling to 100 samples for SHAP value computation. To use all samples, set max_samples=500 when initializing the masker.
100%|===================| 4997/5000 [12:23<00:00]        

SHAP values shape (n_classes, n_samples, n_features): (5, 1000, 40)


## 6a. SHAP Summary Bar Plot (all classes)

In [8]:
# Mean absolute SHAP across all classes and samples → (n_features,)
mean_abs_shap = np.abs(shap_arr).mean(axis=(0, 1))

top_shap_idx   = np.argsort(mean_abs_shap)[::-1][:20]
top_shap_names = [feature_names[i] for i in top_shap_idx]
top_shap_vals  = mean_abs_shap[top_shap_idx]

fig, ax = plt.subplots(figsize=(9, 6))
bars = ax.barh(top_shap_names[::-1], top_shap_vals[::-1], color='darkorange', edgecolor='white')
ax.set_xlabel('Mean |SHAP value|', fontsize=12)
ax.set_title('Top 20 Features by Mean |SHAP| (all classes)', fontsize=13, fontweight='bold')
ax.bar_label(bars, fmt='%.4f', padding=3, fontsize=8)
plt.tight_layout()

shap_path = os.path.join(MODELS_DIR, 'shap_importance.png')
plt.savefig(shap_path, dpi=150)
plt.show()
print(f'Saved \u2192 {shap_path}')

Saved → d:\NIDS\models\shap_importance.png


## 6b. SHAP Bar Chart Per Class

In [9]:
# shap_arr shape: (n_classes, n_samples, n_features)
n_classes = len(CLASS_NAMES)
fig, axes = plt.subplots(1, n_classes, figsize=(4 * n_classes, 5), sharey=False)

for cls_i, (ax, cls_name) in enumerate(zip(axes, CLASS_NAMES)):
    sv_cls     = np.abs(shap_arr[cls_i])   # (n_samples, n_features)
    mean_sv    = sv_cls.mean(axis=0)       # (n_features,)
    top10_idx  = np.argsort(mean_sv)[::-1][:10]
    top10_names = [feature_names[i] for i in top10_idx]
    top10_vals  = mean_sv[top10_idx]

    ax.barh(top10_names[::-1], top10_vals[::-1], color='steelblue', edgecolor='white')
    ax.set_title(cls_name, fontsize=11, fontweight='bold')
    ax.set_xlabel('Mean |SHAP|', fontsize=9)

plt.suptitle('Top 10 SHAP Features per Class', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()

shap_cls_path = os.path.join(MODELS_DIR, 'shap_per_class.png')
plt.savefig(shap_cls_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved \u2192 {shap_cls_path}')

Saved → d:\NIDS\models\shap_per_class.png


## 7. Save Model + Model Info JSON

In [10]:
# ── 7a. Save model ─────────────────────────────────────────────
model_path = os.path.join(MODELS_DIR, 'model.pkl')
joblib.dump(model, model_path)
print(f'\u2713 Model saved   \u2192 {model_path}')

# ── 7b. Build confusion matrix dict for the API ────────────────
cm_dict = {}
for i, actual_cls in enumerate(CLASS_NAMES):
    cm_dict[actual_cls] = {}
    for j, pred_cls in enumerate(CLASS_NAMES):
        cm_dict[actual_cls][pred_cls] = int(cm[i][j])

# ── 7c. Build feature importance list ─────────────────────────
# Always derived directly from model — safe even if Cell 5 was skipped
_importances = model.feature_importances_
feature_importance_list = [
    [feature_names[i], float(_importances[i])]
    for i in np.argsort(_importances)[::-1][:20]
]

# ── 7d. Assemble model_info.json ───────────────────────────────
model_info = {
    'algorithm'         : 'XGBoost (XGBClassifier)',
    'dataset'           : 'NSL-KDD',
    'samples'           : int(len(X_train)),
    'features'          : int(len(feature_names)),
    'train_size'        : f'{len(X_train):,} rows (after SMOTE)',
    'test_accuracy'     : round(float(test_acc), 4),
    'test_f1_weighted'  : round(float(test_f1),  4),
    'val_accuracy'      : round(float(val_acc),  4),
    'confusion_matrix'  : cm_dict,
    'feature_importance': feature_importance_list,
    'classes'           : CLASS_NAMES,
    'mlflow_run_id'     : RUN_ID,
    'hyperparameters'   : {k: (int(v) if isinstance(v, (np.integer,)) else
                               float(v) if isinstance(v, (np.floating,)) else v)
                           for k, v in params.items()},
}

info_path = os.path.join(MODELS_DIR, 'model_info.json')
with open(info_path, 'w') as f:
    json.dump(model_info, f, indent=2)
print(f'\u2713 Model info saved \u2192 {info_path}')

✓ Model saved   → d:\NIDS\models\model.pkl
✓ Model info saved → d:\NIDS\models\model_info.json


## 8. Log Artifacts to MLflow

In [11]:
with mlflow.start_run(run_id=RUN_ID):
    mlflow.xgboost.log_model(model, artifact_path='xgboost_model')
    for fpath in [cm_path, fi_path, shap_path, shap_cls_path, info_path]:
        if os.path.exists(fpath):
            mlflow.log_artifact(fpath)

print('\u2713 All artifacts logged to MLflow')

2026/06/18 12:51:53 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


✓ All artifacts logged to MLflow


## 9. Final Summary

In [12]:
print('=' * 55)
print('  TRAINING COMPLETE')
print('=' * 55)
print(f'  Val Accuracy  : {val_acc:.4f}')
print(f'  Val F1        : {val_f1:.4f}')
print(f'  Test Accuracy : {test_acc:.4f}')
print(f'  Test F1       : {test_f1:.4f}')
print()
print(f'  model.pkl       \u2192 {model_path}')
print(f'  model_info.json \u2192 {info_path}')
print()
print('=' * 55)
print('  Backend is ready. Start FastAPI with:')
print('  cd backend && uvicorn main:app --reload')
print('=' * 55)

  TRAINING COMPLETE
  Val Accuracy  : 0.9992
  Val F1        : 0.9992
  Test Accuracy : 0.8022
  Test F1       : 0.7724

  model.pkl       → d:\NIDS\models\model.pkl
  model_info.json → d:\NIDS\models\model_info.json

  Backend is ready. Start FastAPI with:
  cd backend && uvicorn main:app --reload
